# ETL Pipelines for Analytics

# Introduction

In the previous lesson you *read* from MongoDB: you ran aggregation
pipelines over applicant records to summarise demographics,
quiz-completion rates, and education levels. Every query was
**read-only** — you pulled information out but never wrote anything back.

This lesson closes the loop. You will **extract** documents from
MongoDB, **transform** them by assigning applicants to experimental
groups, and **load** the modified records back into the database. That
three-step rhythm — Extract → Transform → Load — is called **ETL**, and
it is one of the most common workflows in all of data engineering.

There is a second, quieter goal. ETL logic gets re-run for many dates and
many experiments, so copy-pasting four loose functions every day is a
recipe for bugs. You will package the whole pipeline inside a Python
**class** — a `MongoRepository` whose methods hide the messy details
behind a clean interface. That idea (exposing *what* an object does while
hiding *how*) is called **abstraction**, and it is the bridge from
"writing scripts" to "building tools."

> 🎯 **By the end of this notebook you will be able to:**
>
> - Explain the Extract, Transform, Load (ETL) paradigm and why it
>   matters for analytics.
> - Query a MongoDB collection to extract documents that match
>   date-based filters.
> - Transform documents by assigning them to control and treatment
>   groups for an A/B experiment.
> - Update (load) modified documents back into MongoDB.
> - Define a Python class with an `__init__` method and instance methods.
> - Build a reusable `MongoRepository` class that encapsulates the full
>   ETL workflow.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1183284599", h="3298dbabb7", width=700, height=450) 

# 1. Conceptual Foundation

## Project context: what we're trying to build

In Lesson 1 you discovered that roughly **one quarter** of DS Lab
applicants never complete the admissions quiz. That is a real business
problem: an applicant who never finishes the quiz never enters the
program. The product team has a hypothesis — maybe a gentle **reminder
email** nudges some of them to come back and finish.

The only honest way to find out is a **controlled experiment** (an
**A/B test**), where we compare two groups that differ in exactly one
thing — whether they got the email:

| Group | Receives reminder email? | Role |
|---|---|---|
| **Treatment** | ✅ Yes | The intervention we're testing |
| **Control** | ❌ No | The baseline we compare against |

🧠 **Why two groups instead of just emailing everyone?** Because
completion rates drift over time on their own (seasonality, marketing,
word of mouth). If we emailed *everybody* and completion rose, we could
never separate "the email worked" from "completions were rising anyway."
The control group is the counterfactual: it tells us what *would* have
happened without the email.

The experiment must be **repeatable**. Every day a script should: pull
that day's new no-quiz applicants, split them into groups, record the
assignment in the database, and export the treatment-group emails for the
email platform. This lesson builds exactly that pipeline.

## The ETL paradigm

ETL stands for **Extract, Transform, Load** — three deliberately
separate stages:

| Stage | What it does | Here |
|---|---|---|
| **Extract** | Retrieve raw data from a source | Query no-quiz applicants from MongoDB |
| **Transform** | Clean, filter, enrich, or reshape it | Shuffle applicants and label control/treatment |
| **Load** | Write results to a destination | Write the group labels back into MongoDB |

💡 **Why keep the stages separate?** Decoupling makes the pipeline easier
to test, debug, and reuse. If the *source* changes (say, a new database),
only the Extract code changes — the Transform logic that splits people
into groups is untouched. Each stage has one job, so when something
breaks you know where to look.

➡️ The rest of section 1 introduces the four MongoDB / Python tools you
need to implement each stage: **date-range queries** (Extract),
**randomised shuffling** (Transform), **`update_one`** (Load), and
**Python classes** (to bundle it all together).

## Querying MongoDB by date

The experiment runs one day at a time, so Extract needs a **date-range
query**. Here's the subtlety: MongoDB stores `createdAt` as a full
*datetime* (date **and** time), not a bare calendar date. There is no
document whose `createdAt` equals `"2022-05-02"` exactly — they all carry
a time-of-day too. So "everything on 2 May" is not an equality test; it
is the half-open interval from midnight that day up to (but not
including) midnight the next day:

$$
\text{start} \le \texttt{createdAt} < \text{start} + 1\;\text{day}
$$

The interval is **half-open** `[start, end)` on purpose: a document
stamped at exactly midnight on 3 May belongs to the 3rd, not the 2nd, so
the upper bound is strict (`<`, not `≤`). pandas makes building the two
endpoints easy:

In [ ]:
import pandas as pd

# Convert a date string to a datetime object
start = pd.to_datetime("2022-05-02", format="%Y-%m-%d")
# Offset by one day
end = start + pd.DateOffset(days=1)

print("start:", start)
print("end  :", end)
print("type :", type(start))

In PyMongo you express "greater-than-or-equal / less-than" with the
**query operators** `$gte` and `$lt`. (These `$`-prefixed names are
MongoDB operators written inside a Python dict — they are *code*, not
math.)

In [ ]:
# Example query dict (not executed against a database yet)
example_query = {
    "createdAt": {"$gte": start, "$lt": end},
    "admissionsQuiz": "incomplete",
}
print(example_query)

This filter retrieves every document whose `createdAt` falls on exactly
one calendar day **and** whose `admissionsQuiz` field is `"incomplete"`.
The two conditions sit side by side in the dict, so MongoDB applies them
with a logical **AND**.

## Randomised group assignment

Once Extract hands you a list of documents, the Transform step splits
them into two equally-sized groups. The split must be **random** —
otherwise some hidden ordering in the data (sign-up time, alphabetical
name) could quietly load one group with a different kind of applicant and
bias the result. Python's `random` module gives us `shuffle`, which
reorders a list in place; slicing at the midpoint then gives two halves:

In [ ]:
import random

demo_ids = list(range(10))
random.seed(42)
random.shuffle(demo_ids)
print("Shuffled:", demo_ids)

mid = len(demo_ids) // 2
control = demo_ids[:mid]
treatment = demo_ids[mid:]
print("Control  :", control)
print("Treatment:", treatment)

Calling `random.seed(...)` *before* shuffling makes the split
**reproducible** — anyone who runs the code with the same seed gets the
exact same groups. That matters for science: a reviewer must be able to
re-run your pipeline and land on identical assignments.

> 📌 **Tip:** If the list has an odd number of elements, integer division
> (`//`) rounds down, so one group ends up with one extra member. For a
> handful of applicants per day this imbalance is negligible.

## Updating documents in MongoDB

After assigning groups you must **persist** the change — that's the Load
stage. PyMongo's `update_one` modifies a single document and takes two
arguments:

| Argument | Purpose | Example |
|---|---|---|
| **filter** | Identifies *which* document | `{"_id": doc_id}` |
| **update** | Says *what* to change | `{"$set": {"group": "..."}}` |

The `$set` operator adds the named fields or overwrites them if they
already exist; fields you don't mention are left untouched.

In [ ]:
# Demonstration (not connected to a real database)
# result = collection.update_one(
#     {"_id": "abc123"},
#     {"$set": {"inExperiment": True, "group": "no email (control)"}},
# )
# result.matched_count   -> 1 if the filter matched a document
# result.modified_count  -> 1 if the document was actually changed

`update_one` returns a result object with two informative counts:

- `matched_count` — how many documents the **filter** found (0 or 1).
- `modified_count` — how many were **actually changed**.

🔍 These can differ. If you update the same document twice with identical
data, `matched_count` stays `1` but `modified_count` drops to `0` —
MongoDB notices nothing actually changed and reports the write as a no-op.
That distinction becomes a useful sanity check later in the lesson.

## Python classes and abstraction

Querying, shuffling, and updating are three separate skills. Run them by
hand every day and you will eventually call them in the wrong order or
forget an argument. A **Python class** lets you wrap the whole sequence so
the caller only writes:

``` python
repo = MongoRepository()
repo.assign_to_groups("2022-05-02")
```

A class bundles **attributes** (the data an object carries) and
**methods** (the functions that act on that data) into a single object.
The special `__init__` method is the **constructor** — it runs once, when
you create a new instance, and is where you store the object's attributes:

In [ ]:
class Greeter:
    """A minimal class to demonstrate __init__ and methods."""

    def __init__(self, name="World"):
        # `self.name` is an *instance attribute*
        self.name = name

    def greet(self):
        """Return a greeting string."""
        return f"Hello, {self.name}!"


g = Greeter("Data Science")
print(g.greet())
print("Attribute:", g.name)

Three ideas to anchor:

- `self` is the instance itself — every method receives it as its first
  argument, which is how a method reaches the object's own data.
- Attributes set in `__init__` (like `self.name`) are visible to *every*
  method on the object.
- Methods are just functions that happen to live inside the class.

🧠 You already use classes constantly without thinking about it: a pandas
`DataFrame` is a class. Its **attributes** include `.shape` and
`.dtypes`; its **methods** include `.head()` and `.describe()`. Building
`MongoRepository` is the same idea applied to your own ETL pipeline.

## Inspecting objects with `dir`

When a function hands you an unfamiliar object — say the `UpdateResult`
from `update_one` — you don't have to guess what's inside it. The
built-in `dir` function lists every attribute and method an object
exposes:

In [ ]:
# dir() on a simple object to illustrate
print([attr for attr in dir(g) if not attr.startswith("_")])

Filtering out names that start with `_` hides Python's internal
machinery and leaves just the **public interface** — the attributes and
methods you're actually meant to use. `dir` is your first move whenever
you meet a new return type.

> 📚 **Key references for this lesson:**
>
> - [`pymongo.MongoClient`](https://pymongo.readthedocs.io/en/stable/api/pymongo/mongo_client.html#pymongo.mongo_client.MongoClient)
>   — connects to a MongoDB server.
> - [`Collection.find`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.find)
>   — queries documents from a collection.
> - [`Collection.aggregate`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.aggregate)
>   — runs an aggregation pipeline.
> - [`Collection.update_one`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.update_one)
>   — modifies a single document.
> - [`pd.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
>   — converts strings to datetime objects.
> - [`pd.DateOffset`](https://pandas.pydata.org/docs/reference/api/pandas.tseries.offsets.DateOffset.html)
>   — adds a time delta to a datetime.
> - [`random.shuffle`](https://docs.python.org/3/library/random.html#random.shuffle)
>   — shuffles a list in place.
> - [`random.seed`](https://docs.python.org/3/library/random.html#random.seed)
>   — sets the random-number generator state for reproducibility.
> - [`dir`](https://docs.python.org/3/library/functions.html#dir) — lists
>   an object's attributes and methods.
> - [Python classes
>   tutorial](https://docs.python.org/3/tutorial/classes.html) — official
>   guide to defining classes.

# Applied Exercises

## 2. Setup

Note: Import the required libraries and classes. It is highly recommended
to place all imports in a single cell at the beginning of the notebook.

**🛠️ Instruction:** Locate the IP address of the machine running MongoDB
and assign it to the variable `MONGODB_HOST`. Make sure to use a
**string** (i.e., wrap the IP in quotes).

**⚠️ Note:** The IP address is **dynamic** — it may change every time you
start the lab. Always check the current IP before proceeding.

**Code 7.2.2.1**:

In [ ]:
import random

import pandas as pd

from pymongo import MongoClient
from wqulibs.database import reset

MONGODB_HOST = "localhost"

## 3. Connect to MongoDB

### Problem

Before any ETL work can begin, you need a live connection to the MongoDB
collection that holds applicant records. Without this connection, no data
can be extracted or updated.

### Approach

Use `MongoClient` to connect to the local MongoDB instance (host
`"localhost"`, port `27017`). Then navigate to the `"wqu-abtest"`
database and select the `"ds-applicants"` collection. Store the
collection object in a variable called `ds_app` so it can be reused
throughout the notebook.

🔄 **Why `reset(ds_app)`?** This lesson *writes* to the database. If you
re-run cells, yesterday's group labels would still be sitting in the
documents and your counts would drift. `reset` restores the collection to
a clean, known starting state so every run is reproducible.

### Tasks

Create a `MongoClient`, access the `"wqu-abtest"` database and the
`"ds-applicants"` collection, and assign the collection to `ds_app`. Then
call `reset(ds_app)` to restore the database to a known state so your
results are reproducible.

**Code 7.2.3.1**:

In [ ]:
client = MongoClient(host=MONGODB_HOST, port=27017)
ds_app = client["wqu-abtest"]["ds-applicants"]
reset(ds_app)
print("client:", type(client))
print("ds_app:", type(ds_app))

### Checkpoint

In [ ]:
assert "MongoClient" in str(type(client)), (
    f"Expected MongoClient, got {type(client)}"
)
assert str(type(ds_app)).endswith("Collection'>"), (
    f"Expected a Collection, got {type(ds_app)}"
)
print("\u2705 Connection established successfully.")

## 4. Extract — Quiz-Completion Rates and Hypothesis

### Problem

Before designing the experiment you need evidence that a problem actually
exists. Specifically, you need to know what proportion of applicants fail
to complete the admissions quiz. You also need to state a **formal
hypothesis** that the experiment will test — *before* you see the
results, so you can't be accused of fitting the story to the data.

### Approach

Use the `aggregate` method on `ds_app` with a `$group` stage to count
documents by `admissionsQuiz` status (`"complete"` vs `"incomplete"`).
Store the counts in `complete` and `incomplete`, then compute
`prop_incomplete` as the ratio of incomplete applicants to the total.

After examining the rates, write a **null hypothesis** ($H_0$) and an
**alternate hypothesis** ($H_a$) as strings.

🧮 **The two hypotheses, stated precisely.** Every hypothesis test pits
two mutually exclusive claims against each other. Let $p_{\text{treatment}}$
be the quiz-completion rate among applicants who receive the reminder
email and $p_{\text{control}}$ the rate among those who don't. Then:

$$
H_0:\; p_{\text{treatment}} = p_{\text{control}}
\qquad\text{vs.}\qquad
H_a:\; p_{\text{treatment}} > p_{\text{control}}
$$

- The **null hypothesis** $H_0$ is the skeptic's position: the email
  makes *no* difference, so the two completion rates are equal. Any gap we
  observe is just luck of the draw.
- The **alternate hypothesis** $H_a$ is the product team's claim: the
  email *raises* completion, so the treatment rate is higher.

💡 We never "prove" $H_a$ directly. Instead we ask: *if $H_0$ were true,
how surprising would our data be?* If the observed gap would be very
unlikely under $H_0$, we **reject** $H_0$ in favour of $H_a$. Quantifying
that surprise is exactly what the chi-square test in the next lesson does.
For now, your job is just to **state** the hypotheses clearly.

### Tasks

Run an aggregation that groups documents by `admissionsQuiz` and counts
each group. Assign the counts to `complete` and `incomplete`. You should
see two printed numbers.

**Reminder:** The result of `aggregate()` is a **cursor** — a single-use
stream. Once you iterate through it (e.g. with a `for` loop or `list()`),
the data is consumed. If you need the data again, re-run the `aggregate()`
call.

**Code Task 7.2.4.1**:

In [ ]:
# group by admissionsQuiz and count each status
result = ds_app.aggregate(
    [{"$group": {"_id": "$admissionsQuiz", "count": {"$sum": 1}}}]
)
for doc in result:
    if doc["_id"] == "incomplete":
        incomplete = doc["count"]
    else:
        complete = doc["count"]

print("Completed quiz:", complete)
print("Did not complete quiz:", incomplete)

Calculate the proportion of applicants who did not complete the quiz. The
result should be a float between 0 and 1.

**Code 7.2.4.2**:

In [ ]:
total = complete + incomplete
prop_incomplete = incomplete / total
print(
    "Proportion who did not complete quiz:",
    round(prop_incomplete, 2),
)

📊 **Reading the result:** the printed proportion lands near `0.25` —
about **a quarter** of applicants never finish the quiz. That is a large
enough leak to be worth fixing, which is what justifies running the
experiment at all. Now turn that motivation into testable statements.

Write the null and alternate hypotheses as strings. The null hypothesis
states that the email has **no effect**; the alternate states that it
**does** increase quiz completion.

**Code 7.2.4.3**:

In [ ]:
null_hypothesis = (
    "There is no relationship between receiving a reminder"
    " email and completing the admissions quiz. Sending the"
    " email does not increase the completion rate."
)
alternate_hypothesis = (
    "There is a relationship between receiving a reminder"
    " email and completing the admissions quiz. Sending the"
    " email does increase the completion rate."
)
print("H0:", null_hypothesis)
print("Ha:", alternate_hypothesis)

✅ Notice these are written in plain English but map exactly onto the
formal statements above: the null asserts "no relationship / no increase"
($p_{\text{treatment}} = p_{\text{control}}$), the alternate asserts "a
relationship / an increase" ($p_{\text{treatment}} > p_{\text{control}}$).
Writing them as prose now keeps the experiment's goal unambiguous when you
return to test it.

### Checkpoint

In [ ]:
assert isinstance(complete, int) and complete > 0, (
    f"Expected `complete` to be a positive int, got {complete!r}"
)
assert isinstance(incomplete, int) and incomplete > 0, (
    f"Expected `incomplete` to be a positive int, got {incomplete!r}"
)
assert 0 < prop_incomplete < 1, (
    f"Expected proportion between 0 and 1, got {prop_incomplete}"
)
assert isinstance(null_hypothesis, str) and len(null_hypothesis) > 10, (
    "Write a descriptive null hypothesis string."
)
assert isinstance(alternate_hypothesis, str) and len(alternate_hypothesis) > 10, (
    "Write a descriptive alternate hypothesis string."
)
print(f"\u2705 Completion rate: {1 - prop_incomplete:.1%} complete, "
      f"{prop_incomplete:.1%} incomplete.")

## 5. Extract — Filter Applicants by Date

### Problem

The experiment runs daily: each day you need to pull only that day's
no-quiz applicants. You need a reusable function that accepts a date
string and returns the matching documents.

### Approach

Create a function `find_by_date(collection, date_string)` that:

1.  Converts `date_string` to a `Timestamp` with `pd.to_datetime`.
2.  Computes `end` as `start + pd.DateOffset(days=1)`.
3.  Builds a query filtering for `createdAt` in `[start, end)` **and**
    `admissionsQuiz == "incomplete"`.
4.  Calls `collection.find(query)` and returns the results as a `list`.

➡️ This is the **Extract** stage from section 1, now wrapped in a
function so you can call it for any date. Note the half-open interval
`[start, end)` is the date-range query you saw earlier.

### Tasks

Define `find_by_date` following the docstring. The function should return
a list of dictionaries (MongoDB documents).

**Code Task 7.2.5.1**:

In [ ]:
def find_by_date(collection, date_string):
    """Find no-quiz applicants created on a given date.

    Parameters
    ----------
    collection : pymongo.collection.Collection
        Collection to search.
    date_string : str
        Date in '%Y-%m-%d' format, e.g. '2022-06-28'.

    Returns
    -------
    observations : list
        List of matching documents (dictionaries).
    """
    start = pd.to_datetime(date_string, format="%Y-%m-%d")
    end = start + pd.DateOffset(days=1)
    query = {
        "createdAt": {"$gte": start, "$lt": end},
        "admissionsQuiz": "incomplete",
    }
    result = collection.find(query)
    observations = list(result)
    return observations

Call `find_by_date` for 2 May 2022 and inspect the first document. You
should see a list of dictionaries, each representing one applicant.

**Code 7.2.5.2**:

In [ ]:
observations = find_by_date(
    ds_app, # pass the collection
    date_string="2022-05-02") # use "2022-05-02"
print("observations type:", type(observations))
print("observations len:", len(observations))
observations[0]

🔍 **Why wrap the cursor in `list()`?** `collection.find` returns a
*cursor*, which — like the aggregation cursor above — is single-use.
Materialising it into a list lets you check its length, index into it, and
iterate over it more than once, which the next steps all need.

### Checkpoint

In [ ]:
assert isinstance(observations, list), (
    f"Expected a list, got {type(observations)}"
)
assert len(observations) > 0, (
    "Expected at least one observation for 2022-05-02."
)
assert isinstance(observations[0], dict), (
    f"Expected list of dicts, got list of {type(observations[0])}"
)
assert "_id" in observations[0], (
    "Each document should contain an '_id' key."
)
print(f"\u2705 Extracted {len(observations)} no-quiz applicants "
      f"for 2022-05-02.")

## 6. Transform — Assign Applicants to Groups

### Problem

With the day's applicants extracted, you need to randomly split them into
a control group (no email) and a treatment group (email). Each document
must be annotated with `"inExperiment": True` and a `"group"` label so the
assignment is recorded when loaded back into the database.

### Approach

Create a function `assign_to_groups(observations)` that:

1.  Shuffles the list with `random.shuffle` (set `random.seed(42)` first
    for reproducibility).
2.  Finds the midpoint index with integer division.
3.  Labels the first half as `"no email (control)"` and the second half
    as `"email (treatment)"`, adding both `"inExperiment"` and `"group"`
    keys to each document.
4.  Returns the annotated list.

➡️ This is the **Transform** stage. The randomised shuffle from section 1
is what guarantees the two groups are comparable; seeding it keeps the
split reproducible.

### Tasks

Define `assign_to_groups` following the docstring. The function modifies
documents in place and returns the list.

**Code Task 7.2.6.1**:

In [ ]:
def assign_to_groups(observations):
    """Randomly assign observations to control and treatment groups.

    Parameters
    ----------
    observations : list
        List of applicant documents.

    Returns
    -------
    observations : list
        Same list with 'inExperiment' and 'group' keys added.
    """
    random.seed(42)
    random.shuffle(observations)
    # find the midpoint index
    idx = len(observations) // 2
    for doc in observations[:idx]:
        doc["inExperiment"] = True
        doc["group"] = "no email (control)"
    for doc in observations[idx:]:
        doc["inExperiment"] = True
        doc["group"] = "email (treatment)"
    return observations

Call `assign_to_groups` on `observations` and inspect the first document.
You should see the new `"inExperiment"` and `"group"` keys.

**Code 7.2.6.2**:

In [ ]:
observations_assigned = assign_to_groups(observations)
print("observations_assigned type:", type(observations_assigned))
print("observations_assigned len:", len(observations_assigned))
observations_assigned[0]

✅ Each document now carries `"inExperiment": True` and a `"group"`
label. The assignment lives only in this in-memory list for the
moment — section 8 (Load) is where it gets written back to MongoDB.

### Checkpoint

In [ ]:
assert isinstance(observations_assigned, list), (
    f"Expected a list, got {type(observations_assigned)}"
)
assert "inExperiment" in observations_assigned[0], (
    "Documents should have an 'inExperiment' key."
)
assert "group" in observations_assigned[0], (
    "Documents should have a 'group' key."
)
groups = {doc["group"] for doc in observations_assigned}
assert groups == {"no email (control)", "email (treatment)"}, (
    f"Expected two groups, got {groups}"
)
n_control = sum(
    1 for d in observations_assigned
    if d["group"] == "no email (control)"
)
n_treat = sum(
    1 for d in observations_assigned
    if d["group"] == "email (treatment)"
)
assert abs(n_control - n_treat) <= 1, (
    f"Groups should be roughly equal. Control: {n_control}, "
    f"Treatment: {n_treat}"
)
print(f"\u2705 Assigned {n_control} to control and "
      f"{n_treat} to treatment.")

## 7. Transform — Export Treatment Emails

### Problem

The product team needs a CSV file containing the email addresses of
applicants in the treatment group so their email platform can send the
reminder. This file must include today's date in its filename and a
`"tag"` column for tracking.

### Approach

Create a function
`export_treatment_emails(observations_assigned, directory=".")` that:

1.  Puts `observations_assigned` into a `pd.DataFrame`.
2.  Adds a `"tag"` column with the value `"ab-test"`.
3.  Filters for rows where `group == "email (treatment)"`.
4.  Builds a filename using today's date (e.g., `"2022-06-28_ab-test.csv"`).
5.  Saves only the `"email"` and `"tag"` columns to that CSV file.

📌 **Why a dated filename and a `tag` column?** The pipeline runs every
day, so each export needs a unique name to avoid overwriting yesterday's
batch, and the `tag` lets the email platform attribute later sign-ups back
to *this* campaign. Small bookkeeping details like these are what make a
daily pipeline auditable.

### Tasks

Define `export_treatment_emails` and call it. After running the cell you
should see a new CSV file in your working directory.

**Code Task 7.2.7.1**:

In [ ]:
def export_treatment_emails(
    observations_assigned, directory="."
):
    """Export treatment-group emails to a dated CSV file.

    Parameters
    ----------
    observations_assigned : list
        Documents with group assignment.
    directory : str, default='.'
        Output directory for the CSV file.

    Returns
    -------
    None
    """
    df = pd.DataFrame(observations_assigned)
    df["tag"] = "ab-test"  # add a "tag" column
    # filter for treatment group only
    mask = df["group"] == "email (treatment)"
    datestring = pd.Timestamp.now().strftime("%Y-%m-%d")
    filename = directory + "/" + datestring + "_ab-test.csv"
    # save only "email" and "tag" columns
    df.loc[mask, ["email", "tag"]].to_csv(filename, index=False)

export_treatment_emails(observations_assigned)

### Checkpoint

In [ ]:
import os

datestring = pd.Timestamp.now().strftime("%Y-%m-%d")
expected_file = f"./{datestring}_ab-test.csv"
assert os.path.exists(expected_file), (
    f"Expected CSV file '{expected_file}' not found."
)
email_df = pd.read_csv(expected_file)
assert "email" in email_df.columns, (
    f"CSV should have an 'email' column, got {list(email_df.columns)}"
)
assert "tag" in email_df.columns, (
    f"CSV should have a 'tag' column, got {list(email_df.columns)}"
)
assert (email_df["tag"] == "ab-test").all(), (
    "Every row in the 'tag' column should be 'ab-test'."
)
print(f"\u2705 CSV exported with {len(email_df)} treatment emails.")

## 8. Load — Update Records in MongoDB

### Problem

The group assignments exist only in your local Python list. To persist
them you must write the `"inExperiment"` and `"group"` fields back into
MongoDB so that downstream analyses and dashboards can see the
assignments.

### Approach

Start by updating a single document to understand how `update_one` works,
then build a function `update_applicants(collection, observations_assigned)`
that loops over every document and calls `update_one`. The function should
return a summary dictionary with keys `"n"` (matched) and `"nModified"`
(changed).

➡️ This is the **Load** stage — the final third of ETL. You'll first do
one update by hand to see the mechanics, then generalise it into a loop.

### Tasks

Select the first document from `observations_assigned` and extract its
`_id`. Then use `find_one` to verify it exists in the collection.

**Code 7.2.8.1**:

In [ ]:
updated_applicant = observations_assigned[0]
applicant_id = updated_applicant["_id"]
print("applicant_id type:", type(applicant_id))
print("applicant_id:", applicant_id)
ds_app.find_one({"_id": applicant_id})

Use `update_one` to write the modified document back. Then query the same
`_id` again to confirm the `"inExperiment"` and `"group"` fields appear.

**Code 7.2.8.2**:

In [ ]:
result = ds_app.update_one(
    {"_id": applicant_id},  # filter by _id
    {"$set": updated_applicant}, # fields to update
)
print("matched:", result.matched_count)
print("modified:", result.modified_count)

Use `dir` to inspect the `result` object and then access its `raw_result`
attribute. This is a good habit when working with unfamiliar return
types.

**Code 7.2.8.3**:

In [ ]:
# list public attributes (filter out names starting with "_")
print([a for a in dir(result) if not a.startswith("_")])
# Inspect raw_result
print(result.raw_result)

Now define `update_applicants` to loop over all documents and accumulate
the matched and modified counts. Then call it with the full
`observations_assigned` list.

**Code 7.2.8.4**:

In [ ]:
def update_applicants(collection, observations_assigned):
    """Update applicant documents in a MongoDB collection.

    Parameters
    ----------
    collection : pymongo.collection.Collection
        Target collection.
    observations_assigned : list
        Documents with group assignments to write back.

    Returns
    -------
    transaction_result : dict
        Keys 'n' (matched) and 'nModified' (changed).
    """
    n = 0
    n_modified = 0
    for doc in observations_assigned:
        result = collection.update_one(
            {"_id": doc["_id"]}, # filter by document _id
            {"$set": doc}, # update with the full doc
        )
        n += result.matched_count
        n_modified += result.modified_count
    transaction_result = {"n": n, "nModified": n_modified}
    return transaction_result

**Code 7.2.8.5**:

In [ ]:
result = update_applicants(ds_app, observations_assigned)
print("result type:", type(result))
result

> 📌 **Tip:** If you run the update cell a second time, `nModified` will
> be `0` because the documents already have the new fields. This is
> expected — MongoDB only counts a modification when a value actually
> changes (recall the `matched_count` vs `modified_count` distinction from
> section 1).

### Checkpoint

In [ ]:
assert isinstance(result, dict), (
    f"Expected a dict, got {type(result)}"
)
assert "n" in result and "nModified" in result, (
    f"Expected keys 'n' and 'nModified', got {list(result.keys())}"
)
assert result["n"] == len(observations_assigned), (
    f"Expected n={len(observations_assigned)}, got n={result['n']}"
)
print(f"\u2705 Updated {result['n']} documents "
      f"({result['nModified']} modified).")

## 9. Build the MongoRepository Class

### Problem

You now have four standalone functions: `find_by_date`,
`assign_to_groups`, `export_treatment_emails`, and `update_applicants`.
Running the full ETL pipeline requires calling them in the right order
with the right arguments — error-prone and tedious. A class can bundle
everything together so that a single method call performs the entire
workflow.

### Approach

Build a `MongoRepository` class step by step:

1.  **`__init__`** — accepts `client`, `db`, and `collection` strings and
    stores the resolved `pymongo.collection.Collection` as
    `self.collection`.
2.  **`find_by_date`** — same logic as your standalone function but uses
    `self.collection`.
3.  **`update_applicants`** — same logic, using `self.collection`.
4.  **`assign_to_groups`** — combines extraction, assignment, and loading
    into one call: takes a `date_string`, calls `self.find_by_date`,
    shuffles and labels the results, calls `self.update_applicants`, and
    returns the transaction result.

🧱 **The payoff of abstraction.** Notice what happens to the call site.
The four loose functions each needed the collection passed in and had to
be invoked in the correct sequence. The class stores the collection *once*
in `__init__`, and `assign_to_groups` chains Extract → Transform → Load
internally. The caller writes one line — `repo.assign_to_groups(date)` —
and the class guarantees the steps run in order. That is the difference
between a script and a tool.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1183284754", h="3298dbabb7", width=700, height=450) 

### Tasks

Define the full `MongoRepository` class. All four elements (`__init__`,
`find_by_date`, `update_applicants`, `assign_to_groups`) should be in a
single class definition. Refer to the standalone functions you already
wrote for the method bodies.

**Code Task 7.2.9.1**:

In [ ]:
class MongoRepository:
    """Repository class for interacting with MongoDB.

    Parameters
    ----------
    client : pymongo.MongoClient
        Default: MongoClient(host=MONGODB_HOST, port=27017).
    db : str
        Default: 'wqu-abtest'.
    collection : str
        Default: 'ds-applicants'.

    Attributes
    ----------
    collection : pymongo.collection.Collection
        Collection for extraction and loading.
    """

    def __init__(
        self,
        client=MongoClient(host=MONGODB_HOST, port=27017),
        db="wqu-abtest",
        collection="ds-applicants",
    ):
        self.collection = client[db][collection]

    def find_by_date(self, date_string):
        start = pd.to_datetime(date_string, format="%Y-%m-%d")
        end = start + pd.DateOffset(days=1)
        query = {
            "createdAt": {"$gte": start, "$lt": end},
            "admissionsQuiz": "incomplete",
        }
        return list(self.collection.find(query))

    def update_applicants(self, documents):
        n = 0
        n_modified = 0
        for doc in documents:
            result = self.collection.update_one(
                {"_id": doc["_id"]},
                {"$set": doc},
            )
            n += result.matched_count
            n_modified += result.modified_count
        return {"n": n, "nModified": n_modified}

    def assign_to_groups(self, date_string):
        observations = self.find_by_date(date_string)
        random.seed(42)
        random.shuffle(observations)
        idx = len(observations) // 2
        for doc in observations[:idx]:
            doc["inExperiment"] = True
            doc["group"] = "no email (control)"
        for doc in observations[idx:]:
            doc["inExperiment"] = True
            doc["group"] = "email (treatment)"
        return self.update_applicants(observations)

Instantiate the class and verify that the `collection` attribute has the
correct type.

**Code 7.2.9.2**:

In [ ]:
repo = MongoRepository()
print("repo type:", type(repo))
c_test = repo.collection
print("collection type:", type(c_test))

Test `find_by_date` through the repository. Query applicants from **15 May
2022** and verify you get a non-empty list.

**Code 7.2.9.3**:

In [ ]:
may_15_users = repo.find_by_date("2022-05-15")
print("may_15_users type:", type(may_15_users))
print("may_15_users len:", len(may_15_users))
may_15_users[:3]

Test `update_applicants` through the repository by re-uploading the
earlier `observations_assigned`. Since the documents are already up to
date, `nModified` should be `0`.

**Code 7.2.9.4**:

In [ ]:
result = repo.update_applicants(observations_assigned)
print("result type:", type(result))
result

🔍 As predicted, `nModified` comes back `0` here: those documents were
already written in section 8, so MongoDB matches them but changes
nothing. The `matched_count` still equals the number of documents, which
confirms the filter found them all.

Finally, test the full `assign_to_groups` pipeline for **14 May 2022**.
This single call extracts, transforms, and loads in one step.

**Code 7.2.9.5**:

In [ ]:
result = repo.assign_to_groups("2022-05-14")
print("result type:", type(result))
result

### Checkpoint

In [ ]:
assert isinstance(repo, MongoRepository), (
    f"Expected MongoRepository instance, got {type(repo)}"
)
assert hasattr(repo, "collection"), (
    "MongoRepository should have a 'collection' attribute."
)
assert hasattr(repo, "find_by_date"), (
    "MongoRepository should have a 'find_by_date' method."
)
assert hasattr(repo, "update_applicants"), (
    "MongoRepository should have an 'update_applicants' method."
)
assert hasattr(repo, "assign_to_groups"), (
    "MongoRepository should have an 'assign_to_groups' method."
)
# End-to-end test with a fresh date
repo_test = MongoRepository()
test_result = repo_test.assign_to_groups("2022-05-16")
assert isinstance(test_result, dict), (
    f"assign_to_groups should return a dict, got {type(test_result)}"
)
assert test_result["n"] > 0, (
    "Expected at least one matched document for 2022-05-16."
)
may_16_docs = repo_test.find_by_date("2022-05-16")
groups_found = {d.get("group") for d in may_16_docs}
assert "no email (control)" in groups_found, (
    "Expected control group in 2022-05-16 documents."
)
assert "email (treatment)" in groups_found, (
    "Expected treatment group in 2022-05-16 documents."
)
print(f"\u2705 MongoRepository class works end-to-end. "
      f"Assigned {test_result['n']} applicants for 2022-05-16.")

# Wrap-up

In this notebook you:

-   Explored quiz-completion rates and formulated null and alternate
    hypotheses ($H_0: p_{\text{treatment}} = p_{\text{control}}$ vs.
    $H_a: p_{\text{treatment}} > p_{\text{control}}$) for the A/B
    experiment.
-   Built a `find_by_date` function to **extract** no-quiz applicants for
    a specific date from MongoDB.
-   Created an `assign_to_groups` function to **transform** documents by
    randomly splitting applicants into control and treatment groups.
-   Wrote an `export_treatment_emails` function to produce a dated CSV
    file for the email platform.
-   Used `update_one` to **load** group assignments back into MongoDB.
-   Encapsulated the full ETL pipeline inside a `MongoRepository` class,
    learning how to define `__init__`, instance attributes, and methods.
-   Practised inspecting unfamiliar objects with `dir` and `raw_result`.

➡️ **Where this is heading:** you now have applicants split into control
and treatment groups, with the assignments persisted in MongoDB. In the
next lesson you'll run the experiment to completion and use the
**chi-square test of independence** to decide whether the reminder email
produced a statistically significant lift in quiz completion — turning the
hypotheses you wrote here into a quantitative verdict.